In [4]:
#import library

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [5]:
#cheking directory

from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/notebooks


In [6]:
#check OS, CPU, RAM

import os
import platform
import psutil

print("Python:", platform.python_version())
print("OS:", platform.platform())

print("\nCPU:")
print("Logical CPUs:", os.cpu_count())

print("\nRAM:")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1024**3:.2f} GB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GB")

Python: 3.11.16
OS: Linux-7.0.0-30-generic-x86_64-with-glibc2.39

CPU:
Logical CPUs: 8

RAM:
Total RAM: 7.50 GB
Available RAM: 2.64 GB


In [7]:
#check GPU

import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
except FileNotFoundError:
    print("No NVIDIA GPU detected.")

No NVIDIA GPU detected.


In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data:", DATA_RAW)
print("Raw data exists:", DATA_RAW.exists())

Project root: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast
Raw data: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/raw
Raw data exists: True


In [9]:
#check data mb

for file in sorted(DATA_RAW.glob("*.csv")):
    size_mb = file.stat().st_size / 1024**2
    print(f"{file.name:30s} {size_mb:10.2f} MB")

holidays_events.csv                  0.02 MB
items.csv                            0.10 MB
oil.csv                              0.02 MB
sample_submission.csv               38.57 MB
stores.csv                           0.00 MB
test.csv                           120.32 MB
train.csv                         4765.94 MB
transactions.csv                     1.48 MB


In [10]:
#check train.csv

train_path = DATA_RAW / "train.csv"

sample = pd.read_csv(train_path, nrows=100_000)

print(sample.shape)
sample.info(memory_usage="deep")

(100000, 6)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id           100000 non-null  int64  
 1   date         100000 non-null  str    
 2   store_nbr    100000 non-null  int64  
 3   item_nbr     100000 non-null  int64  
 4   unit_sales   100000 non-null  float64
 5   onpromotion  0 non-null       float64
dtypes: float64(2), int64(3), str(1)
memory usage: 5.5 MB


In [11]:
#check total size of csv file

files = [
    "train.csv",
    "test.csv",
    "items.csv",
    "stores.csv",
    "transactions.csv",
    "oil.csv",
    "holidays_events.csv",
]

inventory = []

for filename in files:
    path = DATA_RAW / filename
    
    inventory.append({
        "file": filename,
        "size_mb": path.stat().st_size / 1024**2,
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

print(
    f"Total CSV disk size: "
    f"{inventory_df['size_mb'].sum():.2f} MB"
)

Total CSV disk size: 4887.88 MB


In [12]:
#Dataset scale 

train_path = DATA_RAW / "train.csv"

n_rows = 0
min_date = None
max_date = None
n_stores = set()
n_items = set()
n_series = set()

for chunk in pd.read_csv(
    train_path,
    usecols=["date", "store_nbr", "item_nbr"],
    parse_dates=["date"],
    chunksize=500_000
):
    n_rows += len(chunk)

    chunk_min = chunk["date"].min()
    chunk_max = chunk["date"].max()

    min_date = chunk_min if min_date is None else min(min_date, chunk_min)
    max_date = chunk_max if max_date is None else max(max_date, chunk_max)

    n_stores.update(chunk["store_nbr"].unique())
    n_items.update(chunk["item_nbr"].unique())
    n_series.update(
        zip(chunk["store_nbr"], chunk["item_nbr"])
    )

print("Rows:", f"{n_rows:,}")
print("Stores:", len(n_stores))
print("Items:", len(n_items))
print("Store-item series:", f"{len(n_series):,}")
print("Min date:", min_date)
print("Max date:", max_date)

Rows: 125,497,040
Stores: 54
Items: 4036
Store-item series: 174,685
Min date: 2013-01-01 00:00:00
Max date: 2017-08-15 00:00:00


In [13]:
#Kiểm tra schema

expected_columns = {
    "id",
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "onpromotion"
}

actual_columns = set(sample.columns)

print("Missing:", expected_columns - actual_columns)
print("Unexpected:", actual_columns - expected_columns)

Missing: set()
Unexpected: set()


In [14]:
#Kiểm tra target
print("unit_sales min:", sample["unit_sales"].min())
print("unit_sales max:", sample["unit_sales"].max())
print("Negative unit_sales:", (sample["unit_sales"] < 0).sum())
print("Missing unit_sales:", sample["unit_sales"].isna().sum())

unit_sales min: -27.0
unit_sales max: 608.909
Negative unit_sales: 7
Missing unit_sales: 0


## Scope Decision

**Selected Scope:** Stratified Subset (10 Stores × 1,000 Items) from 2016-06-01 to 2017-08-15.

### Reason & Resource Constraints:
- **Available RAM:** 1.50 GB – 3.08 GB (measured dynamically via psutil; conservative baseline design target: 2.10 GB).
- **Raw Data Ingestion:** 
  - `train.csv` (4,765.94 MB, 125.5M rows) / `train.parquet` (1.0 GB) processed out-of-core via chunked streaming to prevent memory exhaustion during extraction.
- **Estimated Scope & Training Matrix:**
  - Selected series: ~10 stores × ~1,000 items ≈ 10,000 store-item pairs.
  - History window: From 2016-06-01 (post-earthquake, full onpromotion coverage, >140 history days before F1 cutoff) to 2017-08-15.
  - Training matrix scale: ~10,000 series × ~20 origins × 16 days ≈ 3,200,000 rows.
  - Expected feature-engineering memory: ~3.2M rows × 40 features × 4 bytes ≈ 512 MB (~0.5 GB), keeping peak memory consumption ≤ 60% of available RAM.
- **Model Training Constraint:** CPU-only training (8 logical cores, no GPU). Ridge and LightGBM with tree constraints fit comfortably within RAM.
- **Expected Runtime:** ~2–5 minutes per holdout fold on 8 logical CPU cores.

### The Selected Scope Preserves:
- **Temporal ordering:** Retains complete daily time sequences without shuffling.
- **Multiple store-item time series:** Balances fast-moving, regular, and intermittent demand patterns across ~10,000 active series.
- **Promotion variation:** Captures promotion dynamics cleanly after 2016-06-01 where `onpromotion` recording is stable.
- **Store/item heterogeneity:** Stratified sampling across store types (A–E), geographical clusters, and item families.
- **Leakage prevention:** Scope selection is strictly fitted using data prior to the F1 cutoff date (≤ 2017-06-12).

### Limitation:
The selected subset represents ~10,000 time series instead of the full 174,685 store-item population, serving as an engineered, reproducible trade-off between representative analytical validity and local hardware compute constraints.

## 1. Problem Definition

Task:
Forecast daily unit_sales for each store-item combination.

Target:
unit_sales

Grain:
store_nbr × item_nbr × date

Forecast horizon:
16 days

## 2. Research Questions

RQ1.
Can machine-learning models improve forecasting performance
over strong seasonal-naive baselines for a 16-day forecasting horizon?

RQ2.
How much do historical, promotion, calendar, store/item,
event, and economic features contribute to forecasting performance?

RQ3.
Does model performance remain stable across different time periods
and different store/item segments?

## 3. Evaluation Protocol

**HOLDOUT Strategy:**
Rolling-origin / expanding-window time-based holdout. Chronological order is strictly preserved with no random shuffling.

**Split Structure:** 3 HOLDOUT Folds + 1 Internal Independent Holdout Set.

**HOLDOUT Horizon:** 16 days per fold ($h \in \{1, \dots, 16\}$).

| Fold / Window | Train Cutoff ($T$) | Cutoff Day of Week | Evaluation Horizon | Holiday & Contextual Characteristics |
|---|---|---|---|---|
| **F1** | 2017-06-12 | Monday | 2017-06-13 → 2017-06-28 | Baseline holdout window; local holidays only (e.g., 06-23, 06-25). |
| **F2** | 2017-06-28 | Wednesday | 2017-06-29 → 2017-07-14 | Mid-period holdout; local holidays only (07-03 Santo Domingo). |
| **F3** | 2017-07-14 | Friday | 2017-07-15 → 2017-07-30 | Immediate pre-holdout holdout; local/cantonal holidays (Guayaquil foundation: 07-23 → 07-25). |
| **HOLDOUT** | 2017-07-30 | Sunday | 2017-07-31 → 2017-08-15 | **Internal Independent Test Set**; contains a shifted national holiday (*Primer Grito de Independencia*, transferred from 2017-08-10 to Friday 2017-08-11). |

*(Note: The unlabelled Kaggle test set covers 2017-08-16 → 2017-08-31 with cutoff on Tuesday 2017-08-15, reserved exclusively for final submission generation and not used as empirical evidence).*

---

### Protocol Design & Leakage Controls

1. **Cutoff Day-of-Week Shift (F13):**
   - The cutoff day advances across folds (Monday → Wednesday → Friday → Sunday).
   - In any 16-day horizon, exactly two weekdays occur 3 times, while the remaining five occur only 2 times. Error metrics will be reported both in aggregate and broken down by day of the week to avoid evaluation bias caused by weekend weighting differences.

2. **Holiday Regime Asymmetry (F11, F13):**
   - Folds F1–F3 contain only localized/cantonal festivities, whereas the Holdout window features a major official transferred national holiday. Holiday feature efficacy will be analyzed separately during ablation to account for this difference.

3. **Strict Expanding-Window Isolation:**
   - All transformations, target aggregations, and historical encodings are fitted strictly on data $\le T$ of each respective fold.
   - The **HOLDOUT** window remains untouched throughout feature design and hyperparameter tuning; it is evaluated strictly once in Experiment E40.

---

### Metrics
- **Primary Metric:** Normalized Weighted Root Mean Squared Logarithmic Error (**NWRMSLE**), applying a weight of $w = 1.25$ to perishable goods and $w = 1.0$ otherwise:
  $$\text{NWRMSLE} = \sqrt{\frac{\sum w_i (\ln(\hat{y}_i + 1) - \ln(y_i + 1))^2}{\sum w_i}}$$
- **Secondary Metrics:** Mean Absolute Error (**MAE**) for tangible volume error interpretation, and Weighted Absolute Percentage Error (**WAPE**) for business inventory aggregation. MAPE is strictly avoided due to division-by-zero instability on sparse days.
- **Reporting Standard:** Mean $\pm$ standard deviation across F1, F2, and F3, reported globally and segmented by demand type (RQ3).

## 4. Baselines

Gọi `T` là ngày cuối cùng đã quan sát (cutoff), `d` là ngày cần dự báo, `h = d - T`.

B0 — Zero forecast: `ŷ = 0`

B1 — Previous-day forecast: `ŷ = y(T)` (giá trị cuối cùng quan sát được)

B2 — Seasonal-naive 7-day forecast: `ŷ = y(d − 7·⌈h/7⌉)`

B3 — Multi-week seasonal average: `ŷ = mean_{k=0..3} y(d − 7·(⌈h/7⌉ + k))`



## 5. Main Models

Primary ML model:
LightGBM

Alternative / sanity-check model:
HistGradientBoosting

Simple ML baseline:
Ridge Regression


## 6. Main Leakage Risks

1. Future target leakage
   Historical features must only use observations available before
   the forecasting cutoff.

2. Rolling-window leakage
   Rolling statistics must not include the holdout period
   or future observations.

3. Future information from auxiliary tables
   Variables such as transactions, promotions, events, or other
   external information must only be used when they would be known
   at forecast time.

## 7. Information Availability & Pre-registered Feature Groups

### 7.1 Information Availability Boundary (Rule D7)

All inputs must adhere strictly to the temporal boundary defined by the cutoff date $T$:

| Information Source | Available at Cutoff $T$? | Usage Rule & Boundary |
|---|---|---|
| **Calendar Attributes** | Yes | Deterministic variables (day of week, month, day of month, payday proxy) allowed for target date $d$. |
| **Store & Item Metadata** | Yes | Static attributes (`family`, `class`, `perishable`, `city`, `state`, `type`, `cluster`) allowed. Target encodings must fit strictly within fold training window ($\le T$). |
| **`onpromotion(d)`** | Yes (per D6) | Allowed for target date $d$. Historical promotion statistics strictly use observations $\le T$. |
| **Sales Target ($y > T$)** | **No** | **Strictly Prohibited.** Absolute barrier against future leakage. |
| **Sales History ($y \le T$)** | Yes | Permitted only via anchored cutoff features ($T'$). |
| **`transactions.csv`** | Only $\le T$ | Terminated on 2017-08-15. Transactions on forecast date $d$ are unknown; only store-level lag/rolling statistics $\le T$ permitted (or excluded based on E9). |
| **`oil.csv`** | Only $\le T$ | Oil prices after $T$ are future economic information. Values must be forward-filled strictly from past observations $\le T$. |
| **`holidays_events.csv`** | Yes (Pre-scheduled) | Allowed. Joined hierarchically by locale (`National` to all stores, `Regional` to `state`, `Local` to `city`). Transferred holiday rules applied per E8. |

---

### 7.2 Pre-registered Feature Groups & EDA Justification (RQ2 Ablation Design)

To evaluate feature group contributions under RQ2 without post-hoc data mining, all feature sets are pre-registered below with their motivating exploratory data analysis (E#) and leakage-prevention safeguards:

| Group | Feature Set Name | Constituent Features | Motivating EDA (E#) | Leakage Safeguard |
|---|---|---|---|---|
| **G1** | Historical Demand | `last_value` ($y(T')$), rolling means (3, 7, 14, 28, 56, 112 days), rolling median & std (28 days), ratio `mean_7 / mean_56`, same-day-of-week means (past 4 and 8 weeks), `n_days_sold_28`, `days_since_last_sale`, `series_age_days`. | E1 (Trend), E2 (Intermittency), E4 (Autocorrelation & lag strength), E5 (Day-of-week pattern), E11 (Series lifecycle). | All sliding windows and summary metrics terminate strictly at anchor cutoff $T'$. |
| **G2** | Promotion Dynamics | `onpromotion(d)` on target day, `promo_next_16` (sum of scheduled promos across the horizon), `promo_last_14` (historical promo intensity). | E7 (Promotional sales lift and cross-family elasticity). | Future promos derived strictly from advance schedules; historical frequencies restricted to $\le T'$. |
| **G3** | Calendar & Payday | Horizon step $h \in \{1, \dots, 16\}$, `dayofweek(d)`, `day_of_month(d)`, `days_to_payday(d)` (Ecuadorian 15th and end-of-month salary cycle). | E5 (Day-of-week variance), E6 (Payday consumption surge). | Deterministic calendar features known indefinitely in advance. |
| **G4** | Holidays & Events | `is_national_holiday(d)`, `is_local_holiday(d, store)`, `days_to_next_holiday(d)`. | E8 (Holiday lift isolated from regular seasonality). | Pre-scheduled public calendar; locale-matched to prevent cross-regional contamination. |
| **G5** | Store & Item Metadata | Categorical identifiers: `store_nbr`, store `type`, `cluster`, `city`, item `family`, `class`, boolean `perishable`. | E10 (Store and category Pareto heterogeneity). | Static metadata; any target encodings are computed strictly inside fold training splits. |
| **G6** | High-Level Hierarchical Aggregates | Item-level mean sales across all stores (28-day window), `family × store` historical interaction mean (28-day window). | E10 (Hierarchical shrinkage), E11 (Cold-start imputation fallback). | Window aggregations calculated strictly using historical observations $\le T'$. |
| **G7** | Contextual Signals | Store-level transactions (28-day rolling mean $\le T'$), 28-day percentage change in oil price ($\le T'$). | E9 (Macroeconomic and traffic predictive power audit). | Retained conditionally based on E9 outcome; strictly prohibits target-date $d$ values. |

---

### 7.3 Planned Ablation Protocol (RQ2)

Feature importance will be quantified through systematic ablation across all three holdout folds using the locked LightGBM pipeline:
1. **Full Model:** All active groups ($\text{G1} + \text{G2} + \text{G3} + \text{G4} + \text{G5} + \text{G6} \ [+ \text{G7}]$).
2. **Without Promotion:** Drop $\text{G2}$ (measures incremental predictive lift from promotional signals).
3. **Without Calendar & Holidays:** Drop $\text{G3} + \text{G4}$ (evaluates reliance on seasonality and events).
4. **Without Metadata:** Drop $\text{G5}$ (evaluates cross-series knowledge sharing).
5. **Without Hierarchical Aggregates:** Drop $\text{G6}$ (evaluates aggregate stabilization on sparse items).
6. **Without Contextual Signals:** Drop $\text{G7}$ (tests macro signal utility).
7. **History Only:** $\text{G1}$ isolated against baselines B0–B3.

## 8. Stopping Criteria

Model development will stop when:

1. The main model has been evaluated on all 3 predefined folds.
2. Major feature groups have been tested through ablation.
3. Additional tuning no longer produces meaningful improvement
   across holdout folds.
4. No unresolved leakage or reproducibility issue remains.
5. The final experiment can be reproduced from the saved configuration.

## 9. Experiment Log

| Experiment | Date | Change | Reason | Result | Decision |
|---|---|---|---|---|---|
| E00 | 2026-09-29 | Scope | | | |
| E01 | | Baseline | | | |
| ... | | | | | |

*(Note: Make sure to `git commit` and create tag `prereg-v1` after this!)*



## 10. Out of Scope

The project will not:

- use random train/test splitting;
- use future target values to construct forecasting features;
- use the Kaggle leaderboard as scientific evidence;
- interpret promotion coefficients as causal effects;
- claim that observed unit_sales equal true unconstrained demand;
- perform unrestricted hyperparameter search;
- compare against an arbitrary large number of models.

## 11. Test Set Protection

Kaggle test set (16-08 đến 31-08) không có nhãn nên không thể đánh giá cục bộ. Tập **HOLDOUT (2017-07-31 → 2017-08-15)** được định nghĩa là **independent test set** nội bộ.

Nó sẽ không được dùng cho:
- Feature engineering decisions
- Hyperparameter tuning / Model comparison
- Threshold/post-processing selection

Nếu independent test set bị dùng để điều chỉnh mô hình, nó sẽ bị giáng cấp thành holdout set.


## 12. Handling Data Anomalies & Modeling Rules (D4–D8)

To preserve causal integrity, prevent data leakage, and handle structural idiosyncrasies in retail logs, all data pipelines must strictly enforce design rules D4 through D8.

---

### D4. Implicit Zeros & Continuous Calendar Grid (F3, F4, F5)
- **Continuous Grid Construction:** For every active store-item pair in the scope, construct a continuous daily date index spanning from its **first recorded sale date** ($\le T$) up to cutoff $T$. Never backfill zeros prior to an item's initial appearance date.
- **Store-Activity Masking:** Implicit zeros are imputed (`unit_sales = 0.0`) **only on active store-days** (dates where the store recorded at least one transaction/sale). 
- **Store Closure Exclusion:** Store-days with zero total sales across all items (e.g., Christmas Dec 25, New Year Jan 1, or temporary shutdowns) represent closure rather than zero demand. These dates are masked as `NaN` and excluded from both training loss and holdout scoring.
- **Strict Prohibition:** Never apply index- or row-position-based `shift()` or `rolling()` on sparse raw records. All lag transformations must operate on the reconstructed calendar grid.

---

### D5. Target Variable Transformation, Negative Values, & Outliers (F6)
- **Negative Sales Clipping:** Raw returns/refunds (`unit_sales < 0`, accounting for 0.0062% of raw records) are clipped to `0.0` for both training targets and historical lag features. The exact number of clipped records must be logged per fold.
- **Logarithmic Target:** Models are trained on $\ln(1 + y)$ (`log1p(y_clipped)`). Raw predictions are transformed back via $\exp(\hat{y}) - 1$ (`expm1`) and clipped to $\ge 0$. This aligns directly with the RMSLE/NWRMSLE optimization objective.
- **Perishable Weighting:** In gradient boosting and regression objectives, assign `sample_weight = 1.25` to perishable goods (`perishable == 1`), and `1.0` otherwise.
- **Heavy Outlier Retention:** Extreme positive demand spikes (up to 89,440 units) are retained without arbitrary truncation; the `log1p` transformation naturally compresses tail variance without discarding genuine bulk purchases.

---

### D6. Promotional Information & Missing Values (F7)
- **Pre-2014 Missingness:** Raw `onpromotion` contains 17.26% `NaN` values, concentrated entirely prior to mid-2014. The training history window (starting 2016-06-01 under D1) lies completely after this missingness regime.
- **Imputed Zero Bias:** Reconstructed grid cells (unobserved store-item days) lack promotion records and are imputed with `onpromotion = False`. 
  - *Documented Limitation:* This creates an upward observational bias in estimated promotional lift, because days where an item was on promotion but achieved zero sales were omitted from raw logs.
- **Forecast Horizon Availability:** Promotional schedules for the 16-day horizon (`onpromotion(d)` where $d \in [T+1, T+16]$) are treated as known at cutoff $T$ (commercial circulars are scheduled in advance and provided in `test.csv`).
- **Non-Causal Interpretation:** Feature importances and regression coefficients for promotions reflect empirical correlations only, not counterfactual causal treatment effects.

---

### D7. Auxiliary Tables & Information Availability Boundary
Any external information utilized must respect the strict temporal cutoff $T$:

| Source Table | Known at Cutoff $T$? | Usage Rule & Boundary |
|---|---|---|
| **Calendar Attributes** | Yes | Deterministic features (day of week, day of month, payday flag) allowed for target date $d$. |
| **Store & Item Metadata** | Yes | Static attributes (`family`, `class`, `perishable`, `city`, `type`, `cluster`) allowed. Target encodings must fit strictly within fold train ($\le T$). |
| **`onpromotion(d)`** | Yes | Allowed for target day $d$. Historical promo summary statistics strictly use dates $\le T$. |
| **Target Sales ($y > T$)** | **No** | **Strictly Prohibited.** Absolute barrier against future leakage. |
| **`transactions.csv`** | Only $\le T$ | auxiliary file terminates on 2017-08-15. Future transactions for day $d$ are unknown. Only historical lags/rolling stats $\le T$ at store level are permitted, or dropped entirely based on E9. |
| **`oil.csv`** | Only $\le T$ | Oil prices after $T$ are future economic information. Strictly forward-fill oil prices using observations $\le T$. Weekend/holiday gaps forward-filled from past. |
| **`holidays_events.csv`** | Yes (Pre-scheduled) | Allowed. Must join by locale hierarchy: `National` $\rightarrow$ all stores; `Regional` $\rightarrow$ matching `state`; `Local` $\rightarrow$ matching `city`. Transferred holidays (`transferred == True`) map off-days to observed celebration dates (E8). |

---

### D8. Cold-Start Hierarchy & Segment Reporting (F8)
- **Fold-Level Definition:** Any store-item combination with no historical sales transactions recorded $\le T$ is designated as a *cold-start* series (accounting for ~21% of series in test).
- **Baseline Behavior:** Historical baselines B0 through B3 predict `0.0` for cold-start pairs.
- **ML Fallback Strategy:** Machine learning pipelines utilize hierarchical aggregate fallbacks: item-level history across other stores $\rightarrow$ `family × store` aggregate levels $\rightarrow$ `0.0`, or leverage tree-native categorical handling.
- **Segmented Error Analysis:** Cold-start performance must be evaluated and reported in an isolated sub-group (RQ3) rather than obscured inside aggregate metrics.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

# Đọc config từ file params.yaml
with open("../configs/params.yaml", "r") as f:
    config = yaml.safe_load(f)
seed = config["seed"]
scope_cutoff = pd.Timestamp(config["scope_cutoff"])

np.random.seed(seed)
DATA_RAW = Path("../data/raw")

# 1. Chọn scope stores (10 stores) phân tầng theo type và city
stores = pd.read_csv(DATA_RAW / "stores.csv")
# Nhóm theo type và city, lấy ngẫu nhiên 1 mẫu đại diện cho mỗi nhóm
scope_stores = stores.groupby(["type", "city"]).sample(n=1, random_state=seed, replace=True).drop_duplicates()
# Nếu vẫn dư thì lấy đúng 10 store
if len(scope_stores) > 10:
    scope_stores = scope_stores.sample(n=10, random_state=seed)
scope_stores.to_csv("../configs/scope_stores.csv", index=False)
print("Đã lưu configs/scope_stores.csv với", len(scope_stores), "stores.")

# 2. Tính tổng doanh số mỗi item (chỉ lấy <= scope_cutoff)
print("Đang đọc train.csv (sẽ mất khoảng 3-5 phút)...")
item_sales = {}
for chunk in pd.read_csv(
    DATA_RAW / "train.csv", 
    usecols=["date", "item_nbr", "unit_sales"],
    parse_dates=["date"],
    chunksize=5_000_000
):
    # Dùng L2: chỉ lấy dữ liệu <= cutoff
    chunk = chunk[chunk["date"] <= scope_cutoff]
    chunk["unit_sales"] = chunk["unit_sales"].clip(lower=0)
    
    sales = chunk.groupby("item_nbr")["unit_sales"].sum()
    for item, s in sales.items():
        item_sales[item] = item_sales.get(item, 0) + s

item_sales_df = pd.DataFrame(list(item_sales.items()), columns=["item_nbr", "total_sales"])
items = pd.read_csv(DATA_RAW / "items.csv")
items = items.merge(item_sales_df, on="item_nbr", how="left").fillna({"total_sales": 0})

# Chia thành 3 nhóm doanh số (tam phân vị)
items["sales_tertile"] = pd.qcut(items["total_sales"], 3, labels=["low", "medium", "high"], duplicates="drop")

# 3. Chọn scope items (~1000 items) phân tầng
scope_items = items.groupby(["family", "sales_tertile", "perishable"]).sample(frac=0.25, random_state=seed, replace=True).drop_duplicates()
if len(scope_items) > 1000:
    scope_items = scope_items.sample(n=1000, random_state=seed)
scope_items.to_csv("../configs/scope_items.csv", index=False)

print("Đã lưu configs/scope_items.csv với", len(scope_items), "items.")
print("\n--- Phân bố Stores (theo type) ---")
print(scope_stores["type"].value_counts())
print("\n--- Phân bố Items (family - top 5) ---")
print(scope_items["family"].value_counts().head())
print("\n--- Phân bố Items (theo perishable) ---")
print(scope_items["perishable"].value_counts())


Đã lưu configs/scope_stores.csv với 10 stores.
Đang đọc train.csv (sẽ mất khoảng 3-5 phút)...
Đã lưu configs/scope_items.csv với 884 items.

--- Phân bố Stores (theo type) ---
type
C    6
D    3
B    1
Name: count, dtype: int64

--- Phân bố Items (family - top 5) ---
family
GROCERY I    293
BEVERAGES    133
CLEANING      90
PRODUCE       66
DAIRY         50
Name: count, dtype: int64

--- Phân bố Items (theo perishable) ---
perishable
0    673
1    211
Name: count, dtype: int64


In [16]:
import json
import re
import subprocess

notebook_path = "00_scope_and_preregistration.ipynb"
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

count = 0
for cell in nb["cells"]:
    if cell["cell_type"] == "markdown":
        new_source = []
        for line in cell["source"]:
            # Xóa các chuỗi có định dạng [cite: <chữ số>]
            new_line, n = re.subn(r'\[cite:.*?\]', '', line)
            new_source.append(new_line)
            count += n
        cell["source"] = new_source

# Ghi lại file notebook
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print(f"Đã xóa {count} chuỗi [cite: ...] trong notebook.")

# Kiểm tra lại theo đúng tiêu chí PASS của PLAN
result = subprocess.run(["grep", "-c", "cite:", notebook_path], capture_output=True, text=True)
# Lệnh grep trả về exit code 1 (rỗng) nếu không tìm thấy, nên ta xử lý in ra 0
print(f"Số từ 'cite:' còn lại trong file: {result.stdout.strip() if result.stdout.strip() else '0'}")
print("\n🚨 LƯU Ý QUAN TRỌNG: Hãy bấm F5 (Reload/Làm mới trang web) NGAY BÂY GIỜ để Jupyter tải lại file mới lên trình duyệt, tránh việc bạn lưu file đè ngược lại!")


Đã xóa 0 chuỗi [cite: ...] trong notebook.
Số từ 'cite:' còn lại trong file: 7

🚨 LƯU Ý QUAN TRỌNG: Hãy bấm F5 (Reload/Làm mới trang web) NGAY BÂY GIỜ để Jupyter tải lại file mới lên trình duyệt, tránh việc bạn lưu file đè ngược lại!


In [17]:
import json
import psutil
from datetime import datetime

# 1. Tính toán thực tế bằng code (không gõ tay)
n_stores = 10
n_items = 884
n_series = n_stores * n_items
origins = 20
horizon = 16
n_rows = n_series * origins * horizon
n_features = 40
bytes_per_value = 4
ram_features_mb = (n_rows * n_features * bytes_per_value) / (1024 ** 2)

ram_available_gb = psutil.virtual_memory().available / (1024 ** 3)
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 2. Cập nhật thẳng vào notebook
notebook_path = "00_scope_and_preregistration.ipynb"
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for cell in nb["cells"]:
    if cell["cell_type"] == "markdown" and "## Scope Decision" in "".join(cell["source"]):
        new_source = [
            "## Scope Decision\n\n",
            f"**Selected Scope:** Stratified Subset ({n_stores} Stores × {n_items} Items) from 2016-06-01 to 2017-08-15.\n\n",
            "### Reason & Resource Constraints:\n",
            f"- **Available RAM:** {ram_available_gb:.2f} GB (measured at {current_time}).\n",
            "- **Raw Data Ingestion:** \n",
            "  - `train.csv` (4,765.94 MB, 125.5M rows) / `train.parquet` (1.0 GB) processed out-of-core via chunked streaming to prevent memory exhaustion during extraction.\n",
            "- **Estimated Scope & Training Matrix:**\n",
            f"  - Selected series: {n_stores} stores × {n_items} items = {n_series:,} store-item pairs.\n",
            "  - History window: From 2016-06-01 (post-earthquake, full onpromotion coverage, >140 history days before F1 cutoff) to 2017-08-15.\n",
            f"  - Training matrix scale: ~{n_series:,} series × ~{origins} origins × {horizon} days ≈ {n_rows:,} rows.\n",
            f"  - Expected feature-engineering memory: ~{n_rows/1e6:.1f}M rows × {n_features} features × {bytes_per_value} bytes ≈ {ram_features_mb:.0f} MB, keeping peak memory consumption ≤ 60% of available RAM.\n",
            "- **Model Training Constraint:** CPU-only training (8 logical cores, no GPU). Ridge and LightGBM with tree constraints fit comfortably within RAM.\n",
            "- **Expected Runtime:** ~2–5 minutes per holdout fold on 8 logical CPU cores.\n\n",
            "### The Selected Scope Preserves:\n",
            "- **Temporal ordering:** Retains complete daily time sequences without shuffling.\n",
            "- **Multiple store-item time series:** Balances fast-moving, regular, and intermittent demand patterns across active series.\n",
            "- **Promotion variation:** Captures promotion dynamics cleanly after 2016-06-01 where `onpromotion` recording is stable.\n",
            "- **Store/item heterogeneity:** Stratified sampling across store types (A–E), geographical clusters, and item families.\n",
            "- **Leakage prevention:** Scope selection is strictly fitted using data prior to the F1 cutoff date (≤ 2017-06-12).\n\n",
            "### Limitation:\n",
            f"The selected subset represents ~{n_series:,} time series instead of the full 174,685 store-item population, serving as an engineered, reproducible trade-off between representative analytical validity and local hardware compute constraints."
        ]
        cell["source"] = new_source

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print(f"Đã cập nhật tính toán RAM và Scope vào cell markdown 'Scope Decision'.")
print(f"- Số pairs: {n_series:,}")
print(f"- Số dòng ma trận: {n_rows:,}")
print(f"- RAM ước lượng cho features: {ram_features_mb:.0f} MB")
print("\n🚨 LƯU Ý QUAN TRỌNG: Hãy bấm F5 (Reload/Làm mới trang web) để Jupyter tải lại văn bản mới!")


Đã cập nhật tính toán RAM và Scope vào cell markdown 'Scope Decision'.
- Số pairs: 8,840
- Số dòng ma trận: 2,828,800
- RAM ước lượng cho features: 432 MB

🚨 LƯU Ý QUAN TRỌNG: Hãy bấm F5 (Reload/Làm mới trang web) để Jupyter tải lại văn bản mới!
